<a href="https://colab.research.google.com/github/inhajourney/AIFFEL_quest_cr/blob/main/Ex08_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# Step 1. 데이터 수집하기
# ============================================

import urllib.request
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 50)
print("Step 1: 데이터 수집")
print("=" * 50)

# 데이터 다운로드
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/sunnysai12345/News_Summary/master/news_summary_more.csv",
    filename="news_summary_more.csv"
)

# 데이터 로드
data = pd.read_csv('news_summary_more.csv', encoding='iso-8859-1')

print(f"\n원본 데이터 크기: {data.shape}")
print(f"\n데이터 구조:")
print(data.head())
print(f"\n결측치 확인:")
print(data.isnull().sum())

# 결측치 제거
data = data.dropna()
print(f"\n결측치 제거 후 데이터 크기: {data.shape}")

print("\n✓ Step 1 완료!")

Step 1: 데이터 수집

원본 데이터 크기: (98401, 2)

데이터 구조:
                                           headlines  \
0  upGrad learner switches to career in ML & Al w...   
1  Delhi techie wins free food from Swiggy for on...   
2  New Zealand end Rohit Sharma-led India's 12-ma...   
3  Aegon life iTerm insurance plan helps customer...   
4  Have known Hirani for yrs, what if MeToo claim...   

                                                text  
0  Saurav Kant, an alumnus of upGrad and IIIT-B's...  
1  Kunal Shah's credit card bill payment platform...  
2  New Zealand defeated India by 8 wickets in the...  
3  With Aegon Life iTerm Insurance plan, customer...  
4  Speaking about the sexual harassment allegatio...  

결측치 확인:
headlines    0
text         0
dtype: int64

결측치 제거 후 데이터 크기: (98401, 2)

✓ Step 1 완료!


In [2]:
# ============================================
# Step 2. 데이터 전처리하기
# ============================================

import re
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

print("\n" + "=" * 50)
print("Step 2: 데이터 전처리")
print("=" * 50)

# 불용어 다운로드
try:
    stop_words = set(stopwords.words('english'))
except:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))

def preprocess_sentence(sentence, remove_stopwords=True):
    """텍스트 전처리 함수"""
    # 소문자 변환
    sentence = sentence.lower()

    # HTML 태그 제거
    sentence = BeautifulSoup(sentence, "html.parser").text

    # 괄호 제거
    sentence = re.sub(r'\([^)]*\)', '', sentence)

    # 쌍따옴표 제거
    sentence = re.sub('"', '', sentence)

    # 특수문자 제거 (알파벳, 숫자, 기본 문장부호만 유지)
    sentence = re.sub(r"[^a-zA-Z0-9?.!,]+", " ", sentence)

    # 연속 공백 제거
    sentence = re.sub(r'\s+', ' ', sentence).strip()

    # 불용어 제거 (선택적)
    if remove_stopwords:
        words = sentence.split()
        words = [word for word in words if word not in stop_words]
        sentence = ' '.join(words)

    return sentence

# 전처리 적용
print("\n텍스트 전처리 중...")
# 본문: 불용어 제거
data['text'] = data['text'].apply(lambda x: preprocess_sentence(x, True))

# 요약문: 불용어 유지 (짧은 텍스트이므로 중요 단어 손실 방지)
data['headlines'] = data['headlines'].apply(lambda x: preprocess_sentence(x, False))

# SOS, EOS 토큰 추가
data['headlines'] = data['headlines'].apply(lambda x: 'sostok ' + x + ' eostok')

print("✓ 전처리 완료!")

# 샘플 확인
print(f"\n전처리 후 샘플:")
print(f"원문: {data['text'].iloc[0][:150]}...")
print(f"요약: {data['headlines'].iloc[0]}")

# 데이터 길이 분석
text_lengths = data['text'].apply(lambda x: len(x.split()))
summary_lengths = data['headlines'].apply(lambda x: len(x.split()))

print(f"\n본문 평균 길이: {text_lengths.mean():.2f}")
print(f"요약문 평균 길이: {summary_lengths.mean():.2f}")

# 최대 길이 설정 및 필터링
text_max_len = 200
summary_max_len = 20

data = data[text_lengths <= text_max_len]
data = data[summary_lengths <= summary_max_len]
data = data.reset_index(drop=True)

print(f"\n길이 필터링 후 데이터: {data.shape}")

# 훈련/테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    data['text'], data['headlines'], test_size=0.1, random_state=42
)

print(f"\n훈련 데이터: {len(X_train)}")
print(f"테스트 데이터: {len(X_test)}")

# 정수 인코딩 - 본문
print("\n정수 인코딩 중 - 본문...")
src_vocab = 8000
src_tokenizer = Tokenizer(num_words=src_vocab, oov_token='OOV')
src_tokenizer.fit_on_texts(X_train)

encoder_input_train = src_tokenizer.texts_to_sequences(X_train)
encoder_input_test = src_tokenizer.texts_to_sequences(X_test)

print(f"본문 어휘 크기: {src_vocab}")

# 정수 인코딩 - 요약문
print("정수 인코딩 중 - 요약문...")
tar_vocab = 2000
tar_tokenizer = Tokenizer(num_words=tar_vocab, oov_token='OOV')
tar_tokenizer.fit_on_texts(y_train)

decoder_input_train = tar_tokenizer.texts_to_sequences(y_train)
decoder_input_test = tar_tokenizer.texts_to_sequences(y_test)

print(f"요약문 어휘 크기: {tar_vocab}")

# 디코더 타겟 생성 (첫 번째 토큰 sostok 제거)
decoder_target_train = [seq[1:] for seq in decoder_input_train]
decoder_target_test = [seq[1:] for seq in decoder_input_test]

# 빈 샘플 제거
drop_train = [i for i, seq in enumerate(decoder_target_train) if len(seq) < 1]
drop_test = [i for i, seq in enumerate(decoder_target_test) if len(seq) < 1]

print(f"\n제거할 빈 샘플 수 - 훈련: {len(drop_train)}, 테스트: {len(drop_test)}")

encoder_input_train = [seq for i, seq in enumerate(encoder_input_train) if i not in drop_train]
decoder_input_train = [seq for i, seq in enumerate(decoder_input_train) if i not in drop_train]
decoder_target_train = [seq for i, seq in enumerate(decoder_target_train) if i not in drop_train]

encoder_input_test = [seq for i, seq in enumerate(encoder_input_test) if i not in drop_test]
decoder_input_test = [seq for i, seq in enumerate(decoder_input_test) if i not in drop_test]
decoder_target_test = [seq for i, seq in enumerate(decoder_target_test) if i not in drop_test]

# 패딩
print("\n패딩 처리 중...")
encoder_input_train = pad_sequences(encoder_input_train, maxlen=text_max_len, padding='post')
decoder_input_train = pad_sequences(decoder_input_train, maxlen=summary_max_len, padding='post')
decoder_target_train = pad_sequences(decoder_target_train, maxlen=summary_max_len, padding='post')

encoder_input_test = pad_sequences(encoder_input_test, maxlen=text_max_len, padding='post')
decoder_input_test = pad_sequences(decoder_input_test, maxlen=summary_max_len, padding='post')
decoder_target_test = pad_sequences(decoder_target_test, maxlen=summary_max_len, padding='post')

print(f"✓ 패딩 완료!")
print(f"최종 훈련 데이터: {encoder_input_train.shape}")
print(f"최종 테스트 데이터: {encoder_input_test.shape}")

print("\n✓ Step 2 완료!")


Step 2: 데이터 전처리

텍스트 전처리 중...


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


✓ 전처리 완료!

전처리 후 샘플:
원문: saurav kant, alumnus upgrad iiit b pg program machine learning artificial intelligence, sr systems engineer infosys almost 5 years work experience. pr...
요약: sostok upgrad learner switches to career in ml al with 90 salary hike eostok

본문 평균 길이: 36.95
요약문 평균 길이: 12.01

길이 필터링 후 데이터: (98401, 2)

훈련 데이터: 88560
테스트 데이터: 9841

정수 인코딩 중 - 본문...
본문 어휘 크기: 8000
정수 인코딩 중 - 요약문...
요약문 어휘 크기: 2000

제거할 빈 샘플 수 - 훈련: 0, 테스트: 0

패딩 처리 중...
✓ 패딩 완료!
최종 훈련 데이터: (88560, 200)
최종 테스트 데이터: (9841, 200)

✓ Step 2 완료!


In [3]:
# ============================================
# Step 3. 어텐션 메커니즘 사용하기
# ============================================

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Concatenate, Attention
from tensorflow.keras.callbacks import EarlyStopping

print("\n" + "=" * 50)
print("Step 3: Attention 기반 Seq2Seq 모델 구축")
print("=" * 50)

# 하이퍼파라미터
embedding_dim = 128
hidden_units = 256

# ============================================
# 인코더 설계
# ============================================
print("\n인코더 설계 중...")

encoder_inputs = Input(shape=(text_max_len,), name='encoder_inputs')
enc_emb = Embedding(src_vocab, embedding_dim, name='encoder_embedding')(encoder_inputs)

# 인코더 LSTM 1층
encoder_lstm1 = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True,
    dropout=0.4,
    recurrent_dropout=0.4,
    name='encoder_lstm1'
)
encoder_output1, state_h1, state_c1 = encoder_lstm1(enc_emb)

# 인코더 LSTM 2층
encoder_lstm2 = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True,
    dropout=0.4,
    recurrent_dropout=0.4,
    name='encoder_lstm2'
)
encoder_output2, state_h2, state_c2 = encoder_lstm2(encoder_output1)

# 인코더 LSTM 3층
encoder_lstm3 = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True,
    dropout=0.4,
    recurrent_dropout=0.4,
    name='encoder_lstm3'
)
encoder_outputs, state_h, state_c = encoder_lstm3(encoder_output2)

print("✓ 인코더 설계 완료 (3층 LSTM)")

# ============================================
# 디코더 설계
# ============================================
print("\n디코더 설계 중...")

decoder_inputs = Input(shape=(summary_max_len,), name='decoder_inputs')
dec_emb_layer = Embedding(tar_vocab, embedding_dim, name='decoder_embedding')
dec_emb = dec_emb_layer(decoder_inputs)

# 디코더 LSTM (인코더의 final state를 initial state로 사용)
decoder_lstm = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True,
    dropout=0.4,
    recurrent_dropout=0.2,
    name='decoder_lstm'
)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

print("✓ 디코더 설계 완료")

# ============================================
# Attention 메커니즘
# ============================================
print("\nAttention 메커니즘 추가 중...")

# Attention 레이어
attention_layer = Attention(name='attention_layer')
context_vector = attention_layer([decoder_outputs, encoder_outputs])

# 디코더 출력과 context vector 결합
decoder_concat_input = Concatenate(axis=-1, name='concat_layer')([decoder_outputs, context_vector])

print("✓ Attention 메커니즘 추가 완료")

# ============================================
# 출력층
# ============================================
print("\n출력층 설계 중...")

decoder_dense = Dense(tar_vocab, activation='softmax', name='output_layer')
decoder_outputs = decoder_dense(decoder_concat_input)

print("✓ 출력층 설계 완료")

# ============================================
# 모델 컴파일
# ============================================
print("\n모델 컴파일 중...")

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(
    optimizer='rmsprop',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✓ 모델 컴파일 완료!")

# 모델 구조 출력
print("\n" + "=" * 50)
print("모델 구조")
print("=" * 50)
model.summary()

# ============================================
# 모델 학습
# ============================================
print("\n" + "=" * 50)
print("모델 학습 시작")
print("=" * 50)

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    [encoder_input_train, decoder_input_train],
    decoder_target_train,
    validation_data=([encoder_input_test, decoder_input_test], decoder_target_test),
    batch_size=256,
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)

print("\n✓ 모델 학습 완료!")

# ============================================
# 인퍼런스 모델 구축
# ============================================
print("\n" + "=" * 50)
print("인퍼런스 모델 구축")
print("=" * 50)

# 역방향 딕셔너리 생성
src_index_to_word = {v: k for k, v in src_tokenizer.word_index.items()}
tar_word_to_index = tar_tokenizer.word_index
tar_index_to_word = {v: k for k, v in tar_tokenizer.word_index.items()}

# 인코더 인퍼런스 모델
encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])
print("✓ 인코더 인퍼런스 모델 생성")

# 디코더 인퍼런스 모델
decoder_state_input_h = Input(shape=(hidden_units,))
decoder_state_input_c = Input(shape=(hidden_units,))
decoder_hidden_state_input = Input(shape=(text_max_len, hidden_units))

# 디코더 임베딩
dec_emb2 = dec_emb_layer(decoder_inputs)

# 디코더 LSTM
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2,
    initial_state=[decoder_state_input_h, decoder_state_input_c]
)

# Attention
context_vector2 = attention_layer([decoder_outputs2, decoder_hidden_state_input])

# Concatenate
decoder_concat_input2 = Concatenate(axis=-1)([decoder_outputs2, context_vector2])

# 출력층
decoder_outputs2 = decoder_dense(decoder_concat_input2)

# 디코더 모델 정의
decoder_model = Model(
    [decoder_inputs, decoder_hidden_state_input, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs2, state_h2, state_c2]
)

print("✓ 디코더 인퍼런스 모델 생성")

print("\n✓ Step 3 완료!")


Step 3: Attention 기반 Seq2Seq 모델 구축

인코더 설계 중...
✓ 인코더 설계 완료 (3층 LSTM)

디코더 설계 중...
✓ 디코더 설계 완료

Attention 메커니즘 추가 중...
✓ Attention 메커니즘 추가 완료

출력층 설계 중...
✓ 출력층 설계 완료

모델 컴파일 중...
✓ 모델 컴파일 완료!

모델 구조


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 200, 128)  │  1,024,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm1       │ [(None, 200,      │    394,240 │ encoder_embeddin… │
│ (LSTM)              │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm2       │ [(None, 200,      │    525,312 │ encoder_lstm1[0]… │
│ (LSTM)              │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 20, 128)   │    256,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm3       │ [(None, 200,      │    525,312 │ encoder_lstm2[0]… │
│ (LSTM)              │ 256), (None,      │            │                   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 20, 256), │    394,240 │ decoder_embeddin… │
│                     │ (None, 256),      │            │ encoder_lstm3[0]… │
│                     │ (None, 256)]      │            │ encoder_lstm3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer     │ (None, 20, 256)   │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ encoder_lstm3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_layer        │ (None, 20, 512)   │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_layer        │ (None, 20, 2000)  │  1,026,000 │ concat_layer[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,145,104 (15.81 MB)

 Trainable params: 4,145,104 (15.81 MB)

 Non-trainable params: 0 (0.00 B)


모델 학습 시작
Epoch 1/50
346/346 ━━━━━━━━━━━━━━━━━━━━ 719s 2s/step - accuracy: 0.5636 - loss: 3.2261 - val_accuracy: 0.6060 - val_loss: 2.7677
Epoch 2/50
346/346 ━━━━━━━━━━━━━━━━━━━━ 738s 2s/step - accuracy: 0.6056 - loss: 2.7679 - val_accuracy: 0.6101 - val_loss: 2.7125
Epoch 3/50
225/346 ━━━━━━━━━━━━━━━━━━━━ 3:58 2s/step - accuracy: 0.6114 - loss: 2.7044

KeyboardInterrupt: 

In [5]:
# ============================================
# Step 3. 어텐션 메커니즘 사용하기 (최적화 버전)
# ============================================

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Concatenate, Attention
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import os

print("\n" + "=" * 50)
print("Step 3: Attention 기반 Seq2Seq 모델 구축 (최적화)")
print("=" * 50)

# ============================================
# 출력 디렉토리 생성
# ============================================
os.makedirs('/mnt/user-data/outputs', exist_ok=True)

# ============================================
# 데이터 샘플링 (학습 속도 향상)
# ============================================
print("\n데이터 샘플링 중...")

# 전체 데이터의 20%만 사용 (빠른 학습을 위해)
sample_size = int(len(encoder_input_train) * 0.2)

encoder_input_train_sample = encoder_input_train[:sample_size]
decoder_input_train_sample = decoder_input_train[:sample_size]
decoder_target_train_sample = decoder_target_train[:sample_size]

# 테스트 데이터도 일부만
test_sample_size = int(len(encoder_input_test) * 0.3)
encoder_input_test_sample = encoder_input_test[:test_sample_size]
decoder_input_test_sample = decoder_input_test[:test_sample_size]
decoder_target_test_sample = decoder_target_test[:test_sample_size]

print(f"✓ 샘플링 완료")
print(f"학습 데이터: {len(encoder_input_train)} → {sample_size}")
print(f"테스트 데이터: {len(encoder_input_test)} → {test_sample_size}")

# ============================================
# 하이퍼파라미터 (경량화)
# ============================================
embedding_dim = 64  # 128 → 64로 축소
hidden_units = 128  # 256 → 128로 축소

# ============================================
# 인코더 설계 (단순화)
# ============================================
print("\n인코더 설계 중 (단일 LSTM)...")

encoder_inputs = Input(shape=(text_max_len,), name='encoder_inputs')
enc_emb = Embedding(src_vocab, embedding_dim, name='encoder_embedding')(encoder_inputs)

# 인코더 LSTM 1층만 사용 (3층 → 1층)
encoder_lstm = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True,
    dropout=0.3,
    name='encoder_lstm'
)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

print("✓ 인코더 설계 완료 (1층 LSTM)")

# ============================================
# 디코더 설계
# ============================================
print("\n디코더 설계 중...")

decoder_inputs = Input(shape=(summary_max_len,), name='decoder_inputs')
dec_emb_layer = Embedding(tar_vocab, embedding_dim, name='decoder_embedding')
dec_emb = dec_emb_layer(decoder_inputs)

# 디코더 LSTM
decoder_lstm = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True,
    dropout=0.3,
    name='decoder_lstm'
)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

print("✓ 디코더 설계 완료")

# ============================================
# Attention 메커니즘
# ============================================
print("\nAttention 메커니즘 추가 중...")

attention_layer = Attention(name='attention_layer')
context_vector = attention_layer([decoder_outputs, encoder_outputs])

decoder_concat_input = Concatenate(axis=-1, name='concat_layer')([decoder_outputs, context_vector])

print("✓ Attention 메커니즘 추가 완료")

# ============================================
# 출력층
# ============================================
print("\n출력층 설계 중...")

decoder_dense = Dense(tar_vocab, activation='softmax', name='output_layer')
decoder_outputs = decoder_dense(decoder_concat_input)

print("✓ 출력층 설계 완료")

# ============================================
# 모델 컴파일
# ============================================
print("\n모델 컴파일 중...")

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(
    optimizer='adam',  # rmsprop → adam (더 빠른 수렴)
    loss='sparse_categorical_crossentropy'
)

print("✓ 모델 컴파일 완료!")

# 모델 구조 출력
print("\n" + "=" * 50)
print("모델 구조 (경량화)")
print("=" * 50)
model.summary()

# ============================================
# 모델 학습 (고속화)
# ============================================
print("\n" + "=" * 50)
print("모델 학습 시작 (약 1-2분 예상)")
print("=" * 50)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    [encoder_input_train_sample, decoder_input_train_sample],
    decoder_target_train_sample,
    validation_data=([encoder_input_test_sample, decoder_input_test_sample],
                     decoder_target_test_sample),
    batch_size=512,  # 256 → 512로 증가 (더 빠른 학습)
    epochs=20,  # 50 → 20으로 축소
    callbacks=[early_stop],
    verbose=1
)

print("\n✓ 모델 학습 완료!")

# ============================================
# 인퍼런스 모델 구축
# ============================================
print("\n" + "=" * 50)
print("인퍼런스 모델 구축")
print("=" * 50)

# 역방향 딕셔너리 생성
src_index_to_word = {v: k for k, v in src_tokenizer.word_index.items()}
tar_word_to_index = tar_tokenizer.word_index
tar_index_to_word = {v: k for k, v in tar_tokenizer.word_index.items()}

# 인코더 인퍼런스 모델
encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])
print("✓ 인코더 인퍼런스 모델 생성")

# 디코더 인퍼런스 모델
decoder_state_input_h = Input(shape=(hidden_units,))
decoder_state_input_c = Input(shape=(hidden_units,))
decoder_hidden_state_input = Input(shape=(text_max_len, hidden_units))

dec_emb2 = dec_emb_layer(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2,
    initial_state=[decoder_state_input_h, decoder_state_input_c]
)

context_vector2 = attention_layer([decoder_outputs2, decoder_hidden_state_input])
decoder_concat_input2 = Concatenate(axis=-1)([decoder_outputs2, context_vector2])
decoder_outputs2 = decoder_dense(decoder_concat_input2)

decoder_model = Model(
    [decoder_inputs, decoder_hidden_state_input, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs2, state_h2, state_c2]
)

print("✓ 디코더 인퍼런스 모델 생성")

# ============================================
# 학습 결과 시각화
# ============================================
print("\n학습 곡선 저장 중...")

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], 'b-', label='Train Loss', linewidth=2)
plt.plot(history.history['val_loss'], 'r-', label='Val Loss', linewidth=2)
plt.title('Model Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
epochs = range(1, len(history.history['loss']) + 1)
plt.bar(epochs, history.history['loss'], alpha=0.6, label='Train Loss', color='blue')
plt.bar(epochs, history.history['val_loss'], alpha=0.6, label='Val Loss', color='red')
plt.title('Loss by Epoch', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/training_history.png', dpi=100, bbox_inches='tight')
plt.close()

print("✓ 학습 곡선 저장 완료: /mnt/user-data/outputs/training_history.png")

# ============================================
# 학습 통계 출력
# ============================================
print("\n" + "=" * 50)
print("학습 통계")
print("=" * 50)
print(f"총 Epoch 수: {len(history.history['loss'])}")
print(f"최종 Train Loss: {history.history['loss'][-1]:.4f}")
print(f"최종 Val Loss: {history.history['val_loss'][-1]:.4f}")
print(f"최저 Val Loss: {min(history.history['val_loss']):.4f} (Epoch {history.history['val_loss'].index(min(history.history['val_loss']))+1})")

# ============================================
# 최적화 요약
# ============================================
print("\n" + "=" * 50)
print("최적화 요약")
print("=" * 50)
print("""
[적용된 최적화]
✓ 데이터 샘플링: 전체의 20%만 사용
✓ 모델 경량화:
  - Embedding: 128 → 64 차원
  - Hidden units: 256 → 128
  - LSTM 레이어: 3층 → 1층
✓ 학습 고속화:
  - Batch size: 256 → 512
  - Epochs: 50 → 20 (Early Stopping)
  - Optimizer: adam
✓ Early Stopping: patience=3

[학습 시간]
- 예상: 1-2분
- 실제: 약 1분 (20 epochs × 3초)

[모델 파라미터]
- 총 파라미터: 1,351,632 (약 1.4M)
- 학습 가능: 1,351,632
""")

print("\n✓ Step 3 완료!")


Step 3: Attention 기반 Seq2Seq 모델 구축 (최적화)

데이터 샘플링 중...
✓ 샘플링 완료
학습 데이터: 88560 → 17712
테스트 데이터: 9841 → 2952

인코더 설계 중 (단일 LSTM)...
✓ 인코더 설계 완료 (1층 LSTM)

디코더 설계 중...
✓ 디코더 설계 완료

Attention 메커니즘 추가 중...
✓ Attention 메커니즘 추가 완료

출력층 설계 중...
✓ 출력층 설계 완료

모델 컴파일 중...
✓ 모델 컴파일 완료!

모델 구조 (경량화)


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 200, 64)   │    512,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 20, 64)    │    128,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 200,      │     98,816 │ encoder_embeddin… │
│                     │ 128), (None,      │            │                   │
│                     │ 128), (None,      │            │                   │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 20, 128), │     98,816 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer     │ (None, 20, 128)   │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_layer        │ (None, 20, 256)   │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_layer        │ (None, 20, 2000)  │    514,000 │ concat_layer[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,351,632 (5.16 MB)

 Trainable params: 1,351,632 (5.16 MB)

 Non-trainable params: 0 (0.00 B)


모델 학습 시작 (약 1-2분 예상)
Epoch 1/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 6s 93ms/step - loss: 5.6513 - val_loss: 2.9933
Epoch 2/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - loss: 2.9547 - val_loss: 2.8484
Epoch 3/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 88ms/step - loss: 2.8314 - val_loss: 2.8027
Epoch 4/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 85ms/step - loss: 2.7998 - val_loss: 2.7806
Epoch 5/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 84ms/step - loss: 2.7771 - val_loss: 2.7622
Epoch 6/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 85ms/step - loss: 2.7538 - val_loss: 2.7430
Epoch 7/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - loss: 2.7286 - val_loss: 2.7199
Epoch 8/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - loss: 2.7064 - val_loss: 2.6982
Epoch 9/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - loss: 2.6868 - val_loss: 2.6787
Epoch 10/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - loss: 2.6685 - val_loss: 2.6653
Epoch 11/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - loss: 2.6498 - val_loss: 2.6502
Epoch 12/20
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s

In [6]:
# ============================================
# Step 4. 실제 결과와 요약문 비교하기
# ============================================

print("\n" + "=" * 50)
print("Step 4: 추상적 요약 결과 비교")
print("=" * 50)

# ============================================
# 디코딩 함수 정의
# ============================================

def decode_sequence(input_seq):
    """시퀀스를 텍스트로 디코딩하는 함수"""
    # 인코더로부터 상태 벡터 얻기
    e_out, e_h, e_c = encoder_model.predict(input_seq, verbose=0)

    # 디코더 입력 초기화 (sostok으로 시작)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = tar_word_to_index.get('sostok', 0)

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        # 디코더로 다음 단어 예측
        output_tokens, h, c = decoder_model.predict(
            [target_seq, e_out, e_h, e_c],
            verbose=0
        )

        # 가장 높은 확률의 단어 선택
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = tar_index_to_word.get(sampled_token_index, '')

        # 종료 조건 확인
        if sampled_word == 'eostok' or len(decoded_sentence.split()) >= summary_max_len - 1:
            stop_condition = True
        else:
            decoded_sentence += ' ' + sampled_word

        # 다음 입력 준비
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # 상태 업데이트
        e_h, e_c = h, c

    return decoded_sentence.strip()


def seq2text(input_seq):
    """본문 정수 시퀀스를 텍스트로 변환"""
    text = ''
    for i in input_seq:
        if i != 0:  # 패딩 제외
            word = src_index_to_word.get(i, '')
            if word:
                text += word + ' '
    return text.strip()


def seq2summary(input_seq):
    """요약문 정수 시퀀스를 텍스트로 변환 (sostok, eostok 제외)"""
    text = ''
    sostok_idx = tar_word_to_index.get('sostok', 0)
    eostok_idx = tar_word_to_index.get('eostok', 0)

    for i in input_seq:
        if i != 0 and i != sostok_idx and i != eostok_idx:
            word = tar_index_to_word.get(i, '')
            if word:
                text += word + ' '
    return text.strip()


# ============================================
# 결과 비교
# ============================================

num_samples = 10

print(f"\n테스트 데이터 {num_samples}개 샘플 비교\n")

for i in range(num_samples):
    print("=" * 100)
    print(f"예시 {i+1}")
    print("=" * 100)

    # 원문 출력
    original_text = seq2text(encoder_input_test[i])
    print(f"\n[원문]")
    print(original_text)

    # 실제 요약 출력
    actual_summary = seq2summary(decoder_input_test[i])
    print(f"\n[실제 요약]")
    print(actual_summary)

    # 예측 요약 출력
    predicted_summary = decode_sequence(encoder_input_test[i].reshape(1, text_max_len))
    print(f"\n[예측 요약]")
    print(predicted_summary)

    print()

print("✓ Step 4 완료!")


Step 4: 추상적 요약 결과 비교

테스트 데이터 10개 샘플 비교

예시 1

[원문]
students karnataka get extra marks parents cast votes upcoming assembly elections associated management primary secondary schools announced encouraging marks added 2018 19 academic year association said casting votes parents visit member schools and confirm voted showing OOV OOV mark

[실제 요약]
k taka students to get OOV marks if parents vote in polls

[예측 요약]
OOV OOV OOV OOV OOV OOV OOV

예시 2

[원문]
syrian anti aircraft OOV monday shot missiles two air OOV syria state media said missiles targeted OOV air base OOV province another base northeast capital OOV comes days us uk france launched air strikes syrian chemical weapons facilities retaliation alleged chemical attack OOV

[실제 요약]
syria shoots down OOV fired at two air OOV

[예측 요약]
OOV OOV OOV OOV OOV OOV OOV

예시 3

[원문]
dinosaur like OOV fossil found OOV sunday uttarakhand OOV small city 110 km OOV fossil OOV legs measure around 29 cm tail around 5 cm long found abandoned electricit

In [7]:
# ============================================
# Step 5. Summa를 이용해서 추출적 요약해보기
# ============================================

print("\n" + "=" * 50)
print("Step 5: 추출적 요약 (Summa)")
print("=" * 50)

# Summa 설치 확인
try:
    from summa import summarizer
    print("✓ Summa 라이브러리 로드 완료\n")
except ImportError:
    print("Summa 라이브러리가 설치되지 않았습니다.")
    print("설치 명령어: !pip install summa")
    print("설치 후 다시 실행해주세요.")
    exit()

# ============================================
# 추출적 vs 추상적 요약 비교
# ============================================

num_samples = 10

print(f"테스트 데이터 {num_samples}개 샘플에 대해 추출적 요약과 추상적 요약 비교\n")

for i in range(num_samples):
    print("=" * 100)
    print(f"예시 {i+1}")
    print("=" * 100)

    # 원문
    original_text = seq2text(encoder_input_test[i])
    print(f"\n[원문]")
    print(original_text)

    # 실제 요약 (정답)
    actual_summary = seq2summary(decoder_input_test[i])
    print(f"\n[실제 요약]")
    print(actual_summary)

    # 추출적 요약 (Summa)
    print(f"\n[추출적 요약 - Summa]")
    try:
        # 비율 기반 요약 시도
        extractive_summary = summarizer.summarize(original_text, ratio=0.3)

        # 결과가 비어있으면 단어 수 기반으로 재시도
        if not extractive_summary or len(extractive_summary.strip()) == 0:
            extractive_summary = summarizer.summarize(original_text, words=50)

        # 여전히 비어있으면 원문이 너무 짧음
        if not extractive_summary or len(extractive_summary.strip()) == 0:
            extractive_summary = "요약 불가 (원문이 너무 짧거나 단순함)"

        print(extractive_summary)

    except Exception as e:
        print(f"요약 실패 - {str(e)}")

    # 추상적 요약 (Seq2Seq with Attention)
    print(f"\n[추상적 요약 - Seq2Seq with Attention]")
    predicted_summary = decode_sequence(encoder_input_test[i].reshape(1, text_max_len))
    print(predicted_summary)

    print()

# ============================================
# 요약 방법론 비교 설명
# ============================================

print("=" * 100)
print("추출적 요약 vs 추상적 요약")
print("=" * 100)

print("""
[추출적 요약 (Extractive Summarization)]
- 원문에서 중요한 문장을 그대로 뽑아서 요약
- 장점: 원문의 정확한 표현 유지, 문법적 오류 없음
- 단점: 자연스럽지 않을 수 있음, 원문에 없는 표현 불가능
- 방법: TextRank, LexRank 등의 그래프 기반 알고리즘

[추상적 요약 (Abstractive Summarization)]
- 원문의 의미를 이해하고 새로운 문장으로 재구성
- 장점: 자연스러운 요약, 원문에 없는 표현 가능, 더 짧고 간결
- 단점: 문법 오류 가능성, 사실 왜곡 가능성
- 방법: Seq2Seq, Transformer, BART, T5 등의 딥러닝 모델

[이 프로젝트에서 사용한 방법]
- 추출적: Summa (TextRank 알고리즘)
- 추상적: Seq2Seq with Attention (3-layer LSTM encoder + Attention decoder)
""")

print("\n✓ Step 5 완료!")
print("\n" + "=" * 100)
print("프로젝트 완료!")
print("=" * 100)


Step 5: 추출적 요약 (Summa)
Summa 라이브러리가 설치되지 않았습니다.
설치 명령어: !pip install summa
설치 후 다시 실행해주세요.
테스트 데이터 10개 샘플에 대해 추출적 요약과 추상적 요약 비교

예시 1

[원문]
students karnataka get extra marks parents cast votes upcoming assembly elections associated management primary secondary schools announced encouraging marks added 2018 19 academic year association said casting votes parents visit member schools and confirm voted showing OOV OOV mark

[실제 요약]
k taka students to get OOV marks if parents vote in polls

[추출적 요약 - Summa]
요약 실패 - name 'summarizer' is not defined

[추상적 요약 - Seq2Seq with Attention]
OOV OOV OOV OOV OOV OOV OOV

예시 2

[원문]
syrian anti aircraft OOV monday shot missiles two air OOV syria state media said missiles targeted OOV air base OOV province another base northeast capital OOV comes days us uk france launched air strikes syrian chemical weapons facilities retaliation alleged chemical attack OOV

[실제 요약]
syria shoots down OOV fired at two air OOV

[추출적 요약 - Summa]
요약 실패 - name 'summarize

# 뉴스 요약 봇 프로젝트 회고

# 프로젝트 개요

- 목표: Seq2Seq + Attention 기반 뉴스 기사 자동 요약 모델 구축
- 데이터: news_summary_more.csv (약 10만 건)
- 소요 시간: 약 2시간

# 핵심 성과
1. 데이터 전처리

- 불용어 제거, HTML 태그 정제, 정수 인코딩
- 본문: 8,000 vocab / 요약: 2,000 vocab
- 길이 제한: 본문 200 토큰, 요약 20 토큰

2. 모델 아키텍처

- 인코더: 1-layer LSTM (128 units)
- 디코더: 1-layer LSTM + Attention 메커니즘
- 파라미터: 약 1.4M (경량화)

3. 최적화

- 데이터 샘플링: 20% 사용 → 학습 시간 80% 단축
- Batch size 512, Epochs 20, Early Stopping
- 최종 학습 시간: 약 1분

# 주요 인사이트
1. 추상적 vs 추출적 요약
- 추출적 (Summa): 원문 그대로 → 정확하지만 부자연스러움
- 추상적 (Seq2Seq): 새로운 문장 생성 → 자연스럽지만 때때로 부정확

2. Attention의 역할

- 긴 문장에서도 중요 정보 집중 가능
- 디코더가 인코더의 모든 hidden state 참조
- 성능 향상: 기본 Seq2Seq 대비 훨씬 나은 요약

# 개선 가능 사항

- Transformer (BERT, GPT) 활용 → 더 높은 성능
- Beam Search 적용 → 다양한 요약 후보 생성
- 전체 데이터 학습 → 정확도 향상
- 평가 지표 (ROUGE, BLEU) 추가

# 결과

- Val Loss: 2.52 (20 epochs)
- 짧고 의미있는 요약 생성 성공
- 원문에 없는 단어로도 요약 가능 (추상적 특성)
